# Can Trading Signals Survive Selection Bias and Transaction Costs?

This notebook audits the executed industry-signal study. It uses committed transformed outputs and synthetic examples, so it does not download or redistribute the provider return panel. The full [report](../results/published/report.md), [protocol](../research_protocol.json) and [methodology](../docs/methodology.md) define the experiment.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "research_protocol.json").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from alpha_research.backtest import rebalance_cost
from alpha_research.inference import max_statistic_test
protocol = json.loads((ROOT / "research_protocol.json").read_text())
selection = json.loads((ROOT / "results/published/frozen_selection.json").read_text())
print("Frozen candidate:", selection["selected_candidate"])
print("Selection information cutoff:", selection["information_cutoff"])
print("Final period:", protocol["holdout_start"], "to", protocol["holdout_end"])

Frozen candidate: momentum_63_skip5__equal
Selection information cutoff: 2014-12-31
Final period: 2015-01-01 to 2025-12-31


## 1. Check the cost-accounting identity

A target is a fraction of equity after costs. Traded notional is measured on equity before costs. For a gross-one portfolio opened from cash, cost fraction is exactly $\tau/(1+\tau)$. This small distinction prevents leverage drift caused by inconsistent cost accounting.

In [2]:
rate = 10 / 10000
cost, turnover = rebalance_cost(np.zeros(2), np.array([0.5, -0.5]), rate)
print("Entry cost fraction:", cost)
print("Analytic check:", rate / (1 + rate))
print("Traded notional / pre-cost equity:", turnover)
assert np.isclose(cost, rate / (1 + rate))

Entry cost fraction: 0.000999000999001
Analytic check: 0.0009990009990009992
Traded notional / pre-cost equity: 0.999000999001


## 2. Look at the actual selection outcome

The selected candidate is fixed before final-period performance is calculated. We do not replace it with the best hindsight performer.

In [3]:
ledger = pd.read_csv(ROOT / "results/published/research_ledger.csv")
winner = selection["selected_candidate"]
chosen = ledger[ledger.candidate == winner]
print(chosen[["period", "annual_mean", "sharpe", "max_drawdown", "annual_turnover"]].to_string(index=False))
print("The negative result is retained.")

     period  annual_mean    sharpe  max_drawdown  annual_turnover
development     0.008784  0.136813     -0.228143        11.615081
 validation     0.029356  0.326939     -0.223310        11.489908
    holdout    -0.001789 -0.020012     -0.375579        12.271468
The negative result is retained.


![Selection outcome](../results/published/figures/selection_decay.png)

## 3. Reconcile the retained daily ledger

A reviewer can reconstruct equity without raw source data. Minor differences arise from CSV rounding to twelve significant digits.

In [4]:
daily = pd.read_csv(ROOT / "results/published/candidate_daily.csv.gz",
    usecols=["date", "candidate", "period", "net_return", "nav"])
path = daily[(daily.candidate == winner) & (daily.period == "holdout")]
rebuilt = np.cumprod(1 + path.net_return.to_numpy())
error = np.max(np.abs(rebuilt - path.nav.to_numpy()))
print("Daily observations:", len(path))
print("Final net equity:", rebuilt[-1])
print("Maximum ledger reconstruction difference:", error)
assert error < 1e-8

Daily observations: 2766
Final net equity: 0.9384453486372314
Maximum ledger reconstruction difference: 5.170752714889204e-12


## 4. Preserve dependence in the family statistic

A perfect duplicate strategy should not create an additional independent search penalty. Joint row resampling preserves that dependence. This is a synthetic demonstration of the code, not the empirical study result.

In [5]:
rng = np.random.default_rng(71)
sample = rng.normal(size=(300, 3))
a = max_statistic_test(sample, replicates=199, seed=901)
b = max_statistic_test(np.column_stack([sample, sample[:, 0]]), replicates=199, seed=901)
print("Global p, original family:", a["global_p"])
print("Global p, duplicate candidate added:", b["global_p"])
assert np.allclose(a["bootstrap_maxima"], b["bootstrap_maxima"])

Global p, original family: 0.785
Global p, duplicate candidate added: 0.785


## 5. Read the empirical and known-null conclusions separately

The empirical family test uses a stationary bootstrap of market-derived candidate returns. The synthetic experiment has a known zero-mean process and independently calibrated max threshold. They illustrate related questions but are not the same estimator.

In [6]:
family = pd.read_csv(ROOT / "results/published/family_inference.csv")
print(family[family.candidate == winner].to_string(index=False))
print()
null = pd.read_csv(ROOT / "results/published/null_summary.csv")
print(null[["method", "false_positive_rate", "lower", "upper"]].to_string(index=False))

               candidate     hac_t  individual_bootstrap_p  family_adjusted_p
momentum_63_skip5__equal -0.073523                0.508246                1.0

                    method  false_positive_rate    lower    upper
prespecified_single_reject                0.050 0.038130 0.065314
              naive_reject                0.650 0.619915 0.678937
         bonferroni_reject                0.054 0.041621 0.069792
     known_null_max_reject                0.049 0.037261 0.064191


## Audit the inference itself

The best-of-50 illustration does not validate the empirical bootstrap. A separate frozen follow-up compares the original fixed-scale test, within-bootstrap HAC studentization, and an exact Gaussian oracle on fresh null datasets. Inspect every procedure, its Monte Carlo interval, and the mandatory secondary empirical comparison; the primary inference is preserved.

In [7]:
audit = pd.read_csv(ROOT / "results/bootstrap_audit/summary.csv")
print(audit[["observations", "method", "rejection_rate", "wilson_low", "wilson_high"]].to_string(index=False))
comparison = json.loads((ROOT / "results/bootstrap_audit/empirical_comparison.json").read_text())
print("\nPrimary fixed-scale family p:", comparison["fixed_scale_p"])
print("Secondary studentized family p:", comparison["studentized_p"])

 observations         method  rejection_rate  wilson_low  wilson_high
          500    fixed_scale           0.093    0.076524     0.112591
          500    studentized           0.060    0.046896     0.076471
          500 known_gaussian           0.054    0.041621     0.069792
         2766    fixed_scale           0.053    0.040747     0.068675
         2766    studentized           0.051    0.039001     0.066435
         2766 known_gaussian           0.047    0.035527     0.061940

Primary fixed-scale family p: 0.792103948025987
Secondary studentized family p: 0.7861069465267366


![Fresh-seed size audit](../results/bootstrap_audit/bootstrap_size.png)

The audit uses six Gaussian candidates and 499 inner replicates, while the empirical family contains thirteen procedures and uses 2,000 replicates. A favorable calibration cell does not establish uniform validity. Read the [complete audit](../results/bootstrap_audit/report.md).

![Null selection](../results/published/figures/null_selection.png)

## Reproduce or extend the study

```bash
alpha-research --profile full --output results/reproduced
```

A new source vintage requires an explicit `--allow-revised-data` flag. The model omits borrow costs, financing, market impact and actual constituent-level execution. Dollar neutrality does not mean factor neutrality. See [limitations](../docs/limitations.md) before interpreting the historical result as an investment opportunity.